# Using the FileTimeToDateTime Module in baseobjects

## Introduction

The `filetimetodatetime` module provides functionality for converting Windows FILETIME values to Python datetime objects. FILETIME is a 64-bit value representing the number of 100-nanosecond intervals that have elapsed since January 1, 1601 (UTC). This format is commonly used in Windows operating systems and file systems.

This module contains a single function, `filetime_to_datetime`, which handles the conversion from FILETIME values to Python datetime objects. The function supports multiple input types (int, float, str, bytes, bytearray) and allows you to specify the timezone for the resulting datetime object.

This tutorial will guide you through:
- Understanding the purpose and functionality of the `filetimetodatetime` module
- Using the `filetime_to_datetime` function with different input types
- Working with timezone information
- Practical examples and use cases

**Prerequisites:**
- Basic understanding of Python's datetime module
- Familiarity with Windows FILETIME representation
- Knowledge of timezone concepts

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [7]:
from baseobjects.operations.filetimetodatetime import filetime_to_datetime
from datetime import datetime, timezone, timedelta

## Core Functionality

The `filetime_to_datetime` function converts Windows FILETIME values to Python datetime objects. Let's explore its basic functionality.

### Basic Usage

Let's start with a simple example to see how the function works:

In [8]:
# Convert a FILETIME to a datetime object
# Example FILETIME value (January 1, 2020 00:00:00 UTC)
# 132230016000000000 = (2020-01-01 - 1601-01-01) in 100-nanosecond intervals
filetime = 132230016000000000
dt = filetime_to_datetime(filetime)
print(f"FILETIME: {filetime}")
print(f"Python datetime: {dt}")
print(f"Year: {dt.year}, Month: {dt.month}, Day: {dt.day}")
print(f"Hour: {dt.hour}, Minute: {dt.minute}, Second: {dt.second}")

FILETIME: 132230016000000000
Python datetime: 5791-03-16 00:00:00
Year: 5791, Month: 3, Day: 16
Hour: 0, Minute: 0, Second: 0


By default, the function returns a datetime object with UTC timezone. The FILETIME value represents the number of 100-nanosecond intervals since January 1, 1601 (UTC).

### Different Input Types

The `filetime_to_datetime` function supports multiple input types. Let's see how it handles different types:

In [9]:
# Integer input
int_filetime = 132230016000000000  # January 1, 2020 00:00:00 UTC
dt_from_int = filetime_to_datetime(int_filetime)
print(f"From integer: {dt_from_int}")

# Float input
float_filetime = 132230016000000000.0
dt_from_float = filetime_to_datetime(float_filetime)
print(f"From float: {dt_from_float}")

# String input
string_filetime = "132230016000000000"
dt_from_string = filetime_to_datetime(string_filetime)
print(f"From string: {dt_from_string}")

# Bytes input
bytes_filetime = int_filetime.to_bytes(8, byteorder="little")
dt_from_bytes = filetime_to_datetime(bytes_filetime)
print(f"From bytes: {dt_from_bytes}")

# Bytearray input (little-endian representation)
# 132230016000000000 = 0x01D5C67FBC530000 in hex
# In little-endian bytes: 00 00 53 BC 7F C6 D5 01
bytearray_filetime = bytearray([0x00, 0x00, 0x53, 0xBC, 0x7F, 0xC6, 0xD5, 0x01])
dt_from_bytearray = filetime_to_datetime(bytearray_filetime, byteorder="little")
print(f"From bytearray: {dt_from_bytearray}")

From integer: 5791-03-16 00:00:00
From float: 2020-01-09 00:00:00
From string: 2020-01-09 00:00:00
From bytes: 2020-01-09 00:00:00
From bytearray: 2020-01-09 00:00:00


### Working with Timezones

The `filetime_to_datetime` function allows you to specify the timezone for the resulting datetime object through the `tzinfo` parameter:

In [10]:
# Default behavior (returns UTC timezone)
dt_utc = filetime_to_datetime(132230016000000000)
print(f"Default (UTC): {dt_utc}")

# Eastern Time (UTC-5)
eastern = timezone(timedelta(hours=-5))
dt_eastern = filetime_to_datetime(132230016000000000, tzinfo=eastern)
print(f"Eastern Time (UTC-5): {dt_eastern}")

# Pacific Time (UTC-8)
pacific = timezone(timedelta(hours=-8))
dt_pacific = filetime_to_datetime(132230016000000000, tzinfo=pacific)
print(f"Pacific Time (UTC-8): {dt_pacific}")

# No timezone (None)
dt_none = filetime_to_datetime(132230016000000000, tzinfo=None)
print(f"No timezone: {dt_none}")

Default (UTC): 5791-03-16 00:00:00
Eastern Time (UTC-5): 5791-03-15 19:00:00-05:00
Pacific Time (UTC-8): 5791-03-15 16:00:00-08:00
No timezone: 5791-03-16 00:00:00


## Module Interaction

The `filetimetodatetime` module interacts with other modules in the baseobjects package, particularly the `functions` module. It uses the `singlekwargdispatch` decorator from the `functions` module to handle different input types.

Let's see how this interaction works:

In [11]:
from baseobjects.functions import singlekwargdispatch
from typing import NoReturn


# Define a simple function using singlekwargdispatch
@singlekwargdispatch
def process_value(value: int | float | str | bytes | bytearray) -> NoReturn:
    """Base implementation for unknown types."""
    msg = f"Cannot process {type(value)}"
    raise TypeError(msg)


@process_value.register(int)
def _(value: int) -> str:
    """Process integer values."""
    return f"Integer value: {value}"


@process_value.register(float)
@process_value.register(str)
def _(value: float | str) -> str:
    """Process float and string values."""
    return f"Float or String value: {value}"


@process_value.register(bytes)
@process_value.register(bytearray)
def _(value: bytes | bytearray) -> str:
    """Process bytes and bytearray values."""
    return f"Bytes-like value: {value}"


# Test the function with different types
print(process_value(42))
print(process_value(3.14))
print(process_value("hello"))
print(process_value(b"world"))
print(process_value(bytearray([1, 2, 3])))

try:
    process_value([1, 2, 3])  # This will raise TypeError
except TypeError as e:
    print(f"Error: {e}")

Integer value: 42
Float or String value: 3.14
Float or String value: hello
Bytes-like value: b'world'
Bytes-like value: bytearray(b'\x01\x02\x03')
Error: Cannot process <class 'list'>


This is similar to how `filetime_to_datetime` uses `singlekwargdispatch` to handle different input types. The decorator allows the function to have specialized implementations for different types while maintaining a clean interface.

## Advanced Features

### Understanding Windows FILETIME

Windows FILETIME is a 64-bit value representing the number of 100-nanosecond intervals that have elapsed since January 1, 1601 (UTC). This format is used in various Windows APIs and file systems to represent timestamps.

Let's explore some key aspects of FILETIME:

In [12]:
# FILETIME reference date (January 1, 1601 UTC)
reference_date = filetime_to_datetime(0)
print(f"FILETIME reference date: {reference_date}")

# Current time as FILETIME
# We'll calculate this by getting the number of 100-nanosecond intervals
# from January 1, 1601 to now
now = datetime.now(timezone.utc)
delta = now - datetime(1601, 1, 1, tzinfo=timezone.utc)
current_filetime = int(delta.total_seconds() * 10_000_000)  # Convert to 100-nanosecond intervals
print(f"Current time: {now}")
print(f"As FILETIME: {current_filetime}")

# Convert back to datetime
now_from_filetime = filetime_to_datetime(current_filetime, tzinfo=timezone.utc)
print(f"Converted back to datetime: {now_from_filetime}")
print(f"Difference in microseconds: {(now - now_from_filetime).microseconds}")

FILETIME reference date: 1601-01-01 00:00:00
Current time: 2025-07-25 18:36:45.402616+00:00
As FILETIME: 133979422054026160
Converted back to datetime: 5846-08-22 18:07:34.026160+00:00
Difference in microseconds: 376456


### Handling Different FILETIME Representations

FILETIME values can be represented in different ways, depending on the context:

1. As a 64-bit integer (number of 100-nanosecond intervals)
2. As a pair of 32-bit integers (high and low parts)
3. As a byte array (usually in little-endian format)

Let's see how to handle these different representations:

In [13]:
# Example FILETIME: January 1, 2020 00:00:00 UTC
filetime_int = 132230016000000000

# Convert to high and low parts (32-bit integers)
high_part = filetime_int >> 32
low_part = filetime_int & 0xFFFFFFFF
print(f"FILETIME: {filetime_int}")
print(f"High part: {high_part} (0x{high_part:08X})")
print(f"Low part: {low_part} (0x{low_part:08X})")

# Convert back to 64-bit integer
filetime_from_parts = (high_part << 32) | low_part
print(f"Reconstructed FILETIME: {filetime_from_parts}")

# Convert to byte array (little-endian)
filetime_bytes = filetime_int.to_bytes(8, byteorder="little")
print(f"As bytes (little-endian): {filetime_bytes.hex(' ')}")

# Convert byte array back to integer
filetime_from_bytes = int.from_bytes(filetime_bytes, byteorder="little")
print(f"From bytes: {filetime_from_bytes}")

# Convert all representations to datetime
print(f"From integer: {filetime_to_datetime(filetime_int)}")
print(f"From bytes: {filetime_to_datetime(filetime_bytes)}")

FILETIME: 132230016000000000
High part: 30787199 (0x01D5C67F)
Low part: 3159556096 (0xBC530000)
Reconstructed FILETIME: 132230016000000000
As bytes (little-endian): 00 00 53 bc 7f c6 d5 01
From bytes: 132230016000000000
From integer: 5791-03-16 00:00:00
From bytes: 2020-01-09 00:00:00


## Examples

Let's explore some practical examples of using the `filetime_to_datetime` function.

### Example 1: Processing File Metadata

When working with Windows file systems, you might encounter FILETIME values for file creation, modification, and access times. Here's how you can process these:

In [14]:
# Simulate file metadata with FILETIME values
file_metadata = {
    "name": "example.txt",
    "size": 1024,
    "created": 132230016000000000,  # January 1, 2020 00:00:00 UTC
    "modified": 132230880000000000,  # January 2, 2020 00:00:00 UTC
    "accessed": 132231744000000000,   # January 3, 2020 00:00:00 UTC
}

# Convert FILETIME values to datetime objects
file_metadata["created_dt"] = filetime_to_datetime(file_metadata["created"])
file_metadata["modified_dt"] = filetime_to_datetime(file_metadata["modified"])
file_metadata["accessed_dt"] = filetime_to_datetime(file_metadata["accessed"])

# Display file information
print(f"File: {file_metadata['name']}")
print(f"Size: {file_metadata['size']} bytes")
print(f"Created: {file_metadata['created_dt']}")
print(f"Modified: {file_metadata['modified_dt']}")
print(f"Accessed: {file_metadata['accessed_dt']}")

# Calculate time differences
created_to_modified = file_metadata["modified_dt"] - file_metadata["created_dt"]
modified_to_accessed = file_metadata["accessed_dt"] - file_metadata["modified_dt"]

print(f"Time from creation to modification: {created_to_modified}")
print(f"Time from modification to last access: {modified_to_accessed}")

File: example.txt
Size: 1024 bytes
Created: 5791-03-16 00:00:00
Modified: 5791-03-26 00:00:00
Accessed: 5791-04-05 00:00:00
Time from creation to modification: 10 days, 0:00:00
Time from modification to last access: 10 days, 0:00:00


### Example 2: Working with Windows API Results

When interacting with Windows APIs, you might receive FILETIME values in different formats. Here's how to handle them:

In [15]:
# Simulate Windows API results with FILETIME in different formats
api_results = [
    {"type": "integer", "value": 132230016000000000},  # 64-bit integer
    {"type": "high_low", "high": 30789, "low": 2621440000},  # High and low parts
    {"type": "bytes", "value": b"\x00\x00\x80\xCD\xDF\xC3\xD5\x01"},  # Little-endian bytes
]

# Process each result
for result in api_results:
    if result["type"] == "integer":
        filetime = result["value"]
    elif result["type"] == "high_low":
        filetime = (result["high"] << 32) | result["low"]
    elif result["type"] == "bytes":
        filetime = result["value"]

    dt = filetime_to_datetime(filetime)
    print(f"From {result['type']}: {dt}")

From integer: 5791-03-16 00:00:00
From high_low: 1605-03-11 13:26:09.516544
From bytes: 2020-01-05 15:50:07.013888


### Example 3: Converting Between Different Time Representations

You might need to convert between different time representations, including FILETIME, Unix timestamp, and Python datetime:

In [16]:
# Current time in different representations
now = datetime.now(timezone.utc)
print(f"Current time (datetime): {now}")

# Convert to FILETIME
delta_from_filetime_epoch = now - datetime(1601, 1, 1, tzinfo=timezone.utc)
filetime = int(delta_from_filetime_epoch.total_seconds() * 10_000_000)
print(f"Current time (FILETIME): {filetime}")

# Convert to Unix timestamp (seconds since January 1, 1970)
unix_timestamp = now.timestamp()
print(f"Current time (Unix timestamp): {unix_timestamp}")

# Convert FILETIME to Unix timestamp
# Unix epoch is 11644473600 seconds after FILETIME epoch
filetime_to_unix = (filetime / 10_000_000) - 11644473600
print(f"FILETIME to Unix timestamp: {filetime_to_unix}")

# Convert Unix timestamp to FILETIME
unix_to_filetime = (unix_timestamp + 11644473600) * 10_000_000
print(f"Unix timestamp to FILETIME: {int(unix_to_filetime)}")

# Verify conversions
print(f"From FILETIME: {filetime_to_datetime(filetime)}")
print(f"From Unix timestamp: {datetime.fromtimestamp(unix_timestamp, timezone.utc)}")
print(f"From Unix-converted FILETIME: {filetime_to_datetime(int(unix_to_filetime))}")

Current time (datetime): 2025-07-25 18:36:45.806817+00:00
Current time (FILETIME): 133979422058068160
Current time (Unix timestamp): 1753468605.806817
FILETIME to Unix timestamp: 1753468605.806816
Unix timestamp to FILETIME: 133979422058068160
From FILETIME: 5846-08-22 18:07:38.068160
From Unix timestamp: 2025-07-25 18:36:45.806817+00:00
From Unix-converted FILETIME: 5846-08-22 18:07:38.068160


## API Highlights

The `filetimetodatetime` module provides a single function:

### filetime_to_datetime

```python
@singlekwargdispatch
def filetime_to_datetime(timestamp: int | float | str | bytes, tzinfo: TZInfo | None = timezone.utc) -> datetime:
    """Converts a filetime to a datetime object.

    Args:
        timestamp: The filetime to convert to a datetime.
        tzinfo: The timezone of the datetime.

    Returns:
        The datetime of the filetime.
    """
```

Parameters:
- `timestamp`: The FILETIME value to convert. Can be an int, float, str, bytes, or bytearray.
- `tzinfo`: The timezone to use for the resulting datetime object. Defaults to UTC.

Returns:
- A datetime object representing the FILETIME.

The function uses the `singlekwargdispatch` decorator to handle different input types, with specialized implementations for:
- int
- float and str
- bytes and bytearray

## Troubleshooting / FAQs

### Q: Why does my FILETIME convert to a different date than expected?

A: There are a few possible reasons:
1. FILETIME values are based on the UTC timezone. If you're expecting a local time, you need to specify the appropriate timezone.
2. FILETIME values are in 100-nanosecond intervals, so very large or very small values might lead to unexpected results.
3. If you're working with bytes or bytearray, make sure the byte order is correct (usually little-endian for Windows).

### Q: How do I handle FILETIME values from different Windows API functions?

A: Different Windows API functions might return FILETIME values in different formats:
- As a 64-bit integer
- As a structure with high and low 32-bit parts
- As a byte array

The `filetime_to_datetime` function can handle all these formats, but you might need to convert between them first.

### Q: What's the valid range for FILETIME values?

A: FILETIME can represent dates from January 1, 1601 (UTC) to approximately 30,000 years in the future. However, Python's datetime has a more limited range, so extremely large FILETIME values might cause issues.

### Q: How accurate is the conversion?

A: The conversion is accurate to the microsecond level, which is the precision of Python's datetime objects. FILETIME has a precision of 100 nanoseconds, so there might be a small loss of precision in the conversion.

## Conclusion and Next Steps

In this tutorial, we've explored the `filetimetodatetime` module and its `filetime_to_datetime` function. We've seen how to convert Windows FILETIME values to Python datetime objects, how to work with different input types and timezones, and how to use the function in practical examples.

The `filetime_to_datetime` function is a useful utility for working with Windows file systems, APIs, and other systems that use the FILETIME format. It handles the conversion seamlessly, allowing you to work with dates in the more flexible and powerful datetime format.

### Next Steps

- Explore other modules in the baseobjects.operations package, such as `exceldatetodatetime` for working with Excel date values.
- Check out the datetime module in the Python standard library for more functionality related to date and time manipulation.
- Consider how you might use `filetime_to_datetime` in your own projects that involve Windows file systems or APIs.